# Realised-state shared cache demo (PR14)

This notebook demonstrates cross-experiment societal-access reuse with a transactional shared cache and reports reuse/performance telemetry.

In [ ]:
from pathlib import Path
import time
import pandas as pd
from ema_workbench import SequentialEvaluator, MultiprocessingEvaluator, Samplers
from src.caching import load_simulation_caches
from src.adaptation import simulate_asset_damage_recovery_access_breakdown_ema
from src.societal_access import list_societal_metric_names
from config import get_development_config

config = get_development_config()
caches = load_simulation_caches(config['interim_dir'], config['hazard_dir'])
shared_db = config['interim_dir'] / 'societal_realized_state_cache.sqlite'

cache_telemetry = {}
shared_cache_cfg = {
    'enabled': True,
    'backend': 'sqlite',
    'path': str(shared_db),
    'namespace': 'use_case_sample_societal_access',
    'schema_version': '1.0.0',
    'timeout_seconds': 30.0,
    'busy_timeout_ms': 30000,
    'max_retries': 5,
    'retry_backoff_seconds': 0.05,
}

societal_access_config = {
    'pop_grid_gdf': population_for_societal,
    'cell_id_column': 'cell_id',
    'pop_group_columns': {
        'total': 'aantal_inwoners',
        'elderly': 'aantal_inwoners_65_jaar_en_ouder',
        'children': 'aantal_inwoners_0_tot_15_jaar',
    },
    'taxonomy': {'msls': 'electricity', 'hospital': 'hospital'},
    'asset_type_column': 'type',
    'all_functions': ['electricity', 'hospital'],
    'allocation_cache': caches.get('societal_allocation_cache') or {},
    'shared_realized_state_cache_config': shared_cache_cfg,
    'cache_telemetry': cache_telemetry,
}


In [ ]:
def run_ema(evaluator_cls, evaluator_kwargs):
    start = time.time()
    with evaluator_cls(model, **evaluator_kwargs) as evaluator:
        experiments, outcomes = evaluator.perform_experiments(
            scenarios=10,
            policies=policies,
            uncertainty_sampling=Samplers.LHS,
        )
    elapsed = time.time() - start
    return experiments, outcomes, elapsed

# sequential baseline
exp_seq, out_seq, t_seq = run_ema(SequentialEvaluator, {})

# multiprocessing / distributed-style run
exp_mp, out_mp, t_mp = run_ema(MultiprocessingEvaluator, {'n_processes': 4})

print('Sequential seconds:', t_seq)
print('Multiprocessing seconds:', t_mp)
print('Speedup:', (t_seq / t_mp) if t_mp > 0 else float('nan'))
print('Cache telemetry:', cache_telemetry)


In [ ]:
telemetry_df = pd.DataFrame([cache_telemetry]).T.reset_index()
telemetry_df.columns = ['metric', 'value']
telemetry_df
